# VMTF-Net - Chạy trực tiếp bằng Google Colab Extension (VS Code / Antigravity)

**Cách chạy:**
1. Ở góc trên bên phải của file notebook này, bấm **Select Kernel** → chọn **Colab** (chọn GPU như T4 / V100 / A100).
2. Bấm **Run All** để chạy toàn bộ pipeline trên GPU Colab ngay tại đây.
3. *(Tùy chọn)* Chạy Cell cuối nếu bạn muốn mở **Terminal bash tương tác** ngay dưới thanh Terminal của IDE kết nối vào máy ảo GPU Colab này.

In [7]:
# 1. Kiểm tra GPU trên Colab Runtime
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "Đang chạy CPU runtime (hãy đổi Kernel sang GPU nếu muốn train nhanh)"

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [8]:
# 2. Clone / cập nhật repo và cài thư viện bổ sung
import os
%cd /content
if not os.path.exists('/content/vietnamese-multimodal-fake-news-detection'):
    !git clone -q https://github.com/kascik04/vietnamese-multimodal-fake-news-detection.git
else:
    !git -C /content/vietnamese-multimodal-fake-news-detection pull -q
%cd /content/vietnamese-multimodal-fake-news-detection
!pip install -q pyvi imagehash
!git log --oneline -1

/content
/content/vietnamese-multimodal-fake-news-detection
00b737a (HEAD -> main, origin/main, origin/HEAD) bigupdate


In [9]:
# 3. Chuẩn bị dữ liệu mẫu (nếu chưa upload data.zip) để kiểm thử pipeline
import os
from PIL import Image, ImageDraw

os.makedirs('data/images', exist_ok=True)
if not os.path.exists('data/train.csv'):
    with open('data/train.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_001,Bộ Y Tế chính thức bác bỏ tin đồn thất thiệt lan truyền trên mạng xã hội sáng nay.,news_1.jpg,1250,340,120,15,30,0\n'
            'post_002,Cảnh báo khẩn cấp: Phương pháp chữa bách bệnh bằng lá cây chưa qua kiểm chứng khoa học!,news_2.jpg,89000,15400,45000,31150,35600,1\n'
            'post_003,Chính phủ phê duyệt dự án phát triển công nghệ cao và trí tuệ nhân tạo quốc gia.,news_3.jpg,4500,820,910,10,45,0\n'
            'post_004,Sốc: Hình ảnh sinh vật lạ xuất hiện tại vùng biển miền Trung gây hoang mang dư luận.,news_4.jpg,52000,8700,19500,13000,28600,1\n'
        )
if not os.path.exists('data/val.csv'):
    with open('data/val.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_001,Bộ Y Tế chính thức bác bỏ tin đồn thất thiệt lan truyền trên mạng xã hội sáng nay.,news_1.jpg,1250,340,120,15,30,0\n'
            'post_002,Cảnh báo khẩn cấp: Phương pháp chữa bách bệnh bằng lá cây chưa qua kiểm chứng khoa học!,news_2.jpg,89000,15400,45000,31150,35600,1\n'
        )
if not os.path.exists('data/test.csv'):
    with open('data/test.csv', 'w', encoding='utf-8') as f:
        f.write(
            'sample_id,text,image_1_local_path,like_count,comment_count,share_count,angry_count,haha_count,label\n'
            'post_003,Chính phủ phê duyệt dự án phát triển công nghệ cao và trí tuệ nhân tạo quốc gia.,news_3.jpg,4500,820,910,10,45,0\n'
            'post_004,Sốc: Hình ảnh sinh vật lạ xuất hiện tại vùng biển miền Trung gây hoang mang dư luận.,news_4.jpg,52000,8700,19500,13000,28600,1\n'
        )

colors = {'news_1.jpg': (30, 136, 229), 'news_2.jpg': (229, 57, 53), 'news_3.jpg': (67, 160, 71), 'news_4.jpg': (251, 140, 0)}
for name, rgb in colors.items():
    p = os.path.join('data/images', name)
    if not os.path.exists(p):
        img = Image.new('RGB', (224, 224), rgb)
        ImageDraw.Draw(img).rectangle([40, 40, 184, 184], outline=(255, 255, 255), width=4)
        img.save(p)

print('Dữ liệu sẵn sàng:', os.listdir('data'), '| images:', os.listdir('data/images'))

Dữ liệu sẵn sàng: ['train.csv', 'dataset_template.csv', 'kb', 'test.csv', 'val.csv', 'images'] | images: ['news_1.jpg', 'news_4.jpg', 'news_2.jpg', 'news_3.jpg']


In [10]:
# 4. Chạy Offline Unit Tests (nếu có) & Train thử VMTF-Net trên Colab GPU
import os
if os.path.exists('test_step1.py'):
    !python test_step1.py && python test_step2.py && python test_step3.py && python test_step4.py && python test_step5.py

!python -m vmtf_net.train \
    --train_csv data/train.csv \
    --val_csv data/val.csv \
    --test_csv data/test.csv \
    --image_dir data/images \
    --cache_dir /content/cache \
    --fusion_type hybrid_proposed \
    --consistency_lambda 0.1 \
    --social_preset basic \
    --batch_size 2 \
    --epochs 2 \
    --lr 2e-5 \
    --head_lr 1e-4 \
    --n_bootstrap 50 \
    --checkpoint_dir checkpoints/colab_ext_smoke_test \
    --overwrite \
    --no_tqdm


--- Testing VMTFConfig ---
[OK] VMTFConfig initialized:
VMTFConfig (Vietnamese Multimodal Transformer Fusion Network)
  - Device:              cuda (AMP: True, Multi-GPU: True)
  - Text Backbone:       vinai/phobert-base (Max Seq: 256, Pool: cls, Seg: auto)
  - Visual Backbone:     google/vit-base-patch16-224 (Image: 224x224)
  - Social Features (3): ['share_count', 'like_count', 'comment_count']
  - Experiment:          hybrid_proposed_phobert-base_vit-base-patch16-224_social-basic_lambda0.15_seed42
  - Batch Size:          8 x accum 1 (Val: 32)
  - Learning Rates:      Backbone: 2e-05, Head: 0.0001
  - Loss Config:         Lambda Consistency: 0.15, Focal: False, Class weights: none
  - Num Workers:         7

[OK] Config save/load, runtime-field reset, presets and validation passed.

--- Testing VMTFDataset & DataLoader ---
[OK] Generated mock dataset in: /tmp/tmp7fs71vew/data
2026-11-13 15:42:21,797 - WARNING - Dropping 1 rows with missing/invalid labels (only 0/1 accepted).
2026-1

In [11]:
# 5. Xem kết quả train & thông số GPU
import json
import pandas as pd

exp_dir = 'checkpoints/colab_ext_smoke_test'
display(pd.read_csv(f'{exp_dir}/history.csv'))
print('--- run_info.json ---')
print(json.dumps(json.load(open(f'{exp_dir}/run_info.json')), indent=2, ensure_ascii=False))
print('--- test metrics ---')
print(json.dumps(json.load(open(f'{exp_dir}/results.json'))['test']['metrics'], indent=2, ensure_ascii=False))

,epoch,train_total_loss,train_cls_loss,train_consistency_loss,train_gate_entropy,train_accuracy,train_f1_macro,lr_backbone,val_threshold,val_n_samples,...,val_recall_macro,val_precision_fake,val_recall_fake,val_auc_roc,val_pr_auc,val_brier,val_ece,val_loss,val_avg_beta,epoch_seconds
0,1,0.773545,0.719088,0.544566,0.673453,0.25,0.200000,0.000017,0.5,2,...,0.5,0.5,1.0,1.0,1.0,0.383426,0.376495,1.128966,0.573505,3.013571
1,2,1.143512,1.094492,0.490196,0.675054,0.50,0.333333,0.000003,0.5,2,...,0.5,0.5,1.0,1.0,1.0,0.351836,0.479876,1.002922,0.537496,2.559884


--- run_info.json ---
{
  "experiment": "hybrid_proposed_phobert-base_vit-base-patch16-224_social-basic_lambda0.1_seed42",
  "environment": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "torch": "2.14.1+cu130",
    "transformers": "5.22.1",
    "cuda_available": true,
    "gpu_count": 1,
    "gpus": [
      "Tesla T4"
    ],
    "kaggle": true,
    "colab": true
  },
  "parameters": {
    "total": 228489220,
    "trainable": 228489220
  },
  "precision": {
    "amp": true,
    "amp_dtype": "float16",
    "tf32": false,
    "num_workers": 7,
    "gpus_used": 1
  },
  "train_samples": 4,
  "val_samples": 2,
  "test_samples": 2,
  "best_epoch": 2,
  "avg_epoch_seconds": 2.7867275695000444,
  "total_run_seconds": 25.5744002739998,
  "peak_gpu_memory_gb": 4.247576713562012,
  "test_samples_per_second": 5.1815047968710815
}
--- test metrics ---
{
  "threshold": 0.83,
  "n_samples": 2,
  "accuracy": 1.0,
  "f1_macro": 1.0,
  "f1_fake": 1.0,
  "precisio